In [2]:
# Attention and Self-Attention

# head is something different. I think it has query, key and value for each layer? let me see

In [7]:
# bigram.py final
# Putting all things together from gpt-dev2 series through gpt-dev2.14
# Commented "# Added" for lines added just for this code
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 32 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?
max_iters = 3000
eval_interval = 300
learning_rate = 1e-2
device = 'cuda' if torch.cuda.is_available() else 'cpu' # Added
eval_iters = 200
n_embd = 32
# ------------

torch.manual_seed(1337)

# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad() # will not call .backward on these tensors. More memory efficient.
def estimate_loss(): # Added
    out = {}
    model.eval() # this doesn't make any sense, since there are no drop out or batch normalization layers.
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        # torch.arange is basically integers from 0 to T - 1
        pos_emb = self.position_embedding_table(torch.arange(T, device = device)) # T, C
        x = tok_emb + pos_emb # (B, T, C) + (T, C) = (B, T, C)
        # Remember, we are mapping from smaller C = 32 to larger vocab_size = 65
        logits = self.lm_head(x) # (B, T, vocab_size)
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = BigramLanguageModel()
m = model.to(device)

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0:
        losses = estimate_loss() # Added
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# generate from the model
# context = torch.zeros((1, 1), dtype=torch.long, device=device) # Added
# print(decode(m.generate(context, max_new_tokens=10)[0].tolist()))

step 0: train loss 4.4801, val loss 4.4801
step 300: train loss 2.5404, val loss 2.5566
step 600: train loss 2.5160, val loss 2.5335
step 900: train loss 2.4967, val loss 2.5149
step 1200: train loss 2.5106, val loss 2.5254
step 1500: train loss 2.4853, val loss 2.5109
step 1800: train loss 2.4966, val loss 2.5198
step 2100: train loss 2.4949, val loss 2.5100
step 2400: train loss 2.4937, val loss 2.5102
step 2700: train loss 2.5040, val loss 2.5114


# Attention

In [21]:
# version 4: self-attention for single individual head.
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels (dimensions)
x = torch.randn(B,T,C)

# this does a simple average of previous tokens
# previous information and current information mixed together as an average
# this is basic matrix multiplication
tril = torch.tril(torch.ones(T, T)) # lower triangular structure
wei = torch.zeros((T,T)) # initialize affinitey between all tokens to be 0
wei = wei.masked_fill(tril == 0, float('-inf')) # mask out
wei = F.softmax(wei, dim=-1)
out = wei @ x

print(f'{tril=}')
print(f'{wei=}')
print('as we can see all weights are uniformly distributed.')
print('This intuitely means, all tokens find all other tokens occuring before it are equally intersting (which is not usually the case')
print('But want these weights or numbers to be data dependent')
print('For example if I\'m a vowel, I\'m looking for consonants in the past. I want that information to flow to me.')

tril=tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])
wei=tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])
as we can see all weights are uniformly distributed.
This int

# How does self-attention solve information flow between tokens?
Every single node, or every single token at each position, will emit 2 vectors
* query - what am I looking for
* key - what do I contain

affinity: dot product between 'key' and 'query'. My 'query' will dot product with all other 'key's. That dot product now becomes 'wei'

If key and query are aligned, they will interact with very high amount. 
I will get to know about that specific token then any other token.

Let's implement single head of self-attention
head_size is hyper-parameter

In [22]:
# version 4: self-attention!
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

# let's see a single Head perform self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
k = key(x) # (B, T, 16)
q = query(x) # (B, T, 16)
# No communication at.

# multiplying q and k is establishing communication
wei = q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) --> (B, T, T)
# BTW k.transpose(-2, -1) is just transpose last and second-last dimension

# wei should be affinities between query and key

tril = torch.tril(torch.ones(T, T))
# wei = torch.zeros((T,T)) --> This is replaced by data dependant by multiplying q & k
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
out = wei @ x

print(f'{tril=}')
print(f'{wei[0]=}')

tril=tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])
wei[0]=tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)


every single batch element in wei is different at each position. So now these numbers represent affinities.
For e.g. last number is 0.2391 (which might encode info like, hey I'm a vowel and I'm at 8th position. I'm looking (via key) for any consonant upto 4th position. query and key when they dot product, will have high affinity. Through softmax, it is aggregated into my position.)

---

#### finalizing
now that we have wei. There is one more part to a single self attention head
We don't aggregate tokens actually.
There is one more 'value' (similar to key and query)

copy pasting above code cell and updating it to add 'value'

In [26]:
# version 4: self-attention!
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

# let's see a single Head perform self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False) # added

k = key(x) # (B, T, 16)
q = query(x) # (B, T, 16)
# No communication at.

# multiplying q and k is establishing communication
wei = q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) --> (B, T, T)
# BTW k.transpose(-2, -1) is just transpose last and second-last dimension

# wei should be affinities between query and key

tril = torch.tril(torch.ones(T, T))
# wei = torch.zeros((T,T)) --> This is replaced by data dependant by multiplying q & k
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

# added now.
v = value(x)
# out = wei @ x
# intuition behind have another layer 'value' is
# think of x as private information to this token (I have this identity, in 5th position, my info is in x)

## For the purposes of single head
## 'query': here's what I'm interested in 
## 'key': here's what I can provide or I have
## 'value': if you find interesting, here's what I'll communicate with you

# v is the thing that gets' aggregated for the purposes of this head
out = wei @ v

print(f'{tril=}')
print(f'{wei[0]=}')

tril=tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])
wei[0]=tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)


---

* **Attention acts as a communication mechanism** where nodes in a graph aggregate information from connected nodes; for example, in an autoregressive language model with an 8-token block size, the graph is structured so that the first node only points to itself, while the eighth node receives information from all previous nodes and itself.
* **Attention mechanisms have no inherent notion of space** and require positional encoding, which contrasts directly with Convolutional Neural Networks (CNNs) that operate over a very specific spatial layout of information.
* Here we wanted one way communication (from left to right). Because we wanted autoregressive setup. Meaning we want to predict next token. And tokens from future should not communicate with tokens in the past. This is also referred to as '**Decoder**'. There are situations where we want full communication in both directions (from left to right and from right to left), like for e.g. sentiment analysis. That is referred to as '**Encoder**' block. Basically removing `wei = wei.masked_fill(tril == 0, float('-inf'))` will essentially make it an '**Encoder**' block.
* There is also called 'cross-attention'. 'keys', 'queries', and 'values' are coming from same source 'x'. There are situations where 'keys' and 'values' come from whole separate source. Those are nodes on the side. Here we generate just 'queries' from 'x'. In principal, 'attention' is lot more general.
---